In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARGA DE DATOS DESDE GITHUB
# Usamos el enlace "raw" para que pandas pueda leer el contenido directamente
url = "https://raw.github.com/JackEspinoza17-sys/Avance/blob/main/Datasets.sql"

try:
    df = pd.read_csv(url)
    print(">>> Conexión exitosa con el repositorio de GitHub.\n")
except Exception as e:
    print(f">>> Error al conectar: {e}")
    df = pd.read_csv('Datasets.sql') 

# -----------------------------------------------------------------
# 2. LIMPIEZA DE DATOS 
# -----------------------------------------------------------------
# Eliminar registros duplicados
df = df.drop_duplicates()

# Limpiar nombres de columnas (quitar espacios en blanco)
df.columns = df.columns.str.strip()

# Tratar valores nulos en columnas críticas (Inversión y Usuarios)
df['Inversion_MUSD'] = df['Inversion_MUSD'].fillna(0)
df['Usuarios_Millones'] = df['Usuarios_Millones'].fillna(df['Usuarios_Millones'].median())

# -----------------------------------------------------------------
# 3. VALIDACIÓN DE DATOS
# -----------------------------------------------------------------
# Asegurar que el año sea coherente (ej. no mayor a 2026)
anio_actual = 2026
df = df[df['Anio_Fundacion'] <= anio_actual]

# Validar que los ingresos/inversión no sean negativos
df = df[df['Inversion_MUSD'] >= 0]

# Convertir tipos de datos para asegurar cálculos precisos
df['Inversion_MUSD'] = pd.to_numeric(df['Inversion_MUSD'])

# -----------------------------------------------------------------
# 4. ESTUDIO ESTADÍSTICO
# -----------------------------------------------------------------
print("--- MÉTRICAS ESTADÍSTICAS DEL SECTOR ---")
# Resumen general de las variables numéricas
print(df[['Inversion_MUSD', 'Usuarios_Millones']].describe())

# Cálculo de la cuota de mercado relativa (Market Share estimando por inversión)
total_inversion = df['Inversion_MUSD'].sum()
df['Share_Inversion'] = (df['Inversion_MUSD'] / total_inversion) * 100

print("\n--- TOP 5 EMPRESAS POR INVERSIÓN ---")
print(df[['Empresa', 'Inversion_MUSD', 'Share_Inversion']].nlargest(5, 'Inversion_MUSD'))

# -----------------------------------------------------------------
# 5. VISUALIZACIÓN 4
# -----------------------------------------------------------------
plt.figure(figsize=(10, 6))
plot = sns.barplot(data=df.nlargest(10, 'Inversion_MUSD'), 
                   x='Inversion_MUSD', y='Empresa', palette='magma')

plt.title('Top 10 Empresas de Software e IA (Inversión en MUSD)', fontsize=14)
plt.xlabel('Millones de USD')
plt.ylabel('Empresa')

# Añadir etiquetas de valor en las barras
for i in plot.containers:
    plot.bar_label(i, padding=3)

plt.tight_layout()
plt.show()